# Task 03A — full Colab study

Select an NVIDIA GPU runtime. NUTS and vectorized prediction use it; official SciPy MAP fitting is profiled and may remain on CPU. `RUN_FULL=False` makes **Run all** safe. Setup installs a clean, isolated package directory under `/content` and executes every scientific command with global site packages disabled, so it does not rely on Colab's broken `venv`/`ensurepip` support or import compiled packages into the notebook kernel. The notebook never pushes to GitHub.

In [ ]:
REPO_URL = 'https://github.com/PaulsonLab/energy-inference-bo.git'
REPO_REF = 'main'  # recommended; setup records the resolved immutable SHA
ACCELERATOR = 'gpu'  # 'gpu' or 'cpu'
RUN_SMOKE = False
RUN_FULL = False

In [ ]:
import os, pathlib, shutil, subprocess, sys
assert (3, 11) <= sys.version_info[:2] <= (3, 12), f'Python {sys.version.split()[0]} is unsupported'
repo = pathlib.Path('/content/energy-inference-bo')
if repo.exists():
    if not (repo/'.git').exists(): raise RuntimeError(f'{repo} exists but is not a Git checkout; delete it manually, then rerun setup.')
    print('Reusing existing checkout; rebuilding only the isolated scientific package directory.')
    subprocess.run(['git','fetch','origin','main'],cwd=repo,check=True)
else:
    subprocess.run(['git','clone','--filter=blob:none',REPO_URL,str(repo)],check=True)
if REPO_REF == 'main':
    checkout_ref = 'origin/main'
else:
    probe = subprocess.run(['git','rev-parse','--verify',f'{REPO_REF}^{{commit}}'],cwd=repo,text=True,capture_output=True)
    if probe.returncode: raise RuntimeError('REPO_REF must be main or a commit already reachable from main; use main and record the printed SHA.')
    checkout_ref = probe.stdout.strip()
subprocess.run(['git','checkout','--detach',checkout_ref],cwd=repo,check=True)
sha=subprocess.check_output(['git','rev-parse','HEAD'],cwd=repo,text=True).strip(); print('Git SHA:',sha)
if ACCELERATOR not in {'cpu','gpu'}: raise ValueError("ACCELERATOR must be 'cpu' or 'gpu'")
if ACCELERATOR == 'gpu':
    subprocess.run(['nvidia-smi'],check=True)
    os.environ['JAX_PLATFORMS']='cuda'; os.environ['XLA_PYTHON_CLIENT_PREALLOCATE']='false'
else: os.environ['JAX_PLATFORMS']='cpu'
os.environ['JAX_ENABLE_X64']='true'
# Colab's system Python currently cannot run ensurepip, so avoid `venv` entirely.
# pip writes all study wheels to this directory; it never changes the notebook kernel.
packages_dir = pathlib.Path('/content/task03a-packages')
if packages_dir.exists(): shutil.rmtree(packages_dir)
packages_dir.mkdir()
# --ignore-installed guarantees that every requirement is copied into packages_dir,
# even when Colab happens to expose a conflicting global distribution.
subprocess.run([sys.executable,'-m','pip','install','--no-cache-dir','-q','--ignore-installed','--target',str(packages_dir),'-r',str(repo/'requirements.txt'),'pytest==9.1.1'],check=True)
if ACCELERATOR == 'gpu': subprocess.run([sys.executable,'-m','pip','install','--no-cache-dir','-q','--ignore-installed','--target',str(packages_dir),'--upgrade','--force-reinstall','-c',str(repo/'requirements.txt'),'jax[cuda12]==0.9.2'],check=True)
scientific_env = os.environ.copy()
scientific_env['PYTHONPATH'] = str(packages_dir) + os.pathsep + str(repo/'src')
scientific_env['PYTHONNOUSERSITE'] = '1'
scientific_env['TASK03A_PACKAGES'] = str(packages_dir)
scientific_python = [sys.executable, '-S']  # disables Colab global site-packages
os.chdir(repo)

In [ ]:
import json
def runtime_info():
    check = """import json, os, sys; import botorch, gpytorch, jax, numpy, numpyro, scipy, torch; modules={'botorch':botorch,'gpytorch':gpytorch,'jax':jax,'numpy':numpy,'numpyro':numpyro,'scipy':scipy,'torch':torch}; root=os.environ['TASK03A_PACKAGES']; origins={name: str(module.__file__) for name,module in modules.items()}; assert all(path.startswith(root + os.sep) for path in origins.values()), f'non-isolated import: {origins}'; jax.config.update('jax_enable_x64', True); print(json.dumps({'python': sys.version, 'numpy': numpy.__version__, 'scipy': scipy.__version__, 'torch': torch.__version__, 'botorch': botorch.__version__, 'gpytorch': gpytorch.__version__, 'jax': jax.__version__, 'numpyro': numpyro.__version__, 'jax_enable_x64': bool(jax.config.jax_enable_x64), 'jax_backend': jax.default_backend(), 'jax_devices': [str(x) for x in jax.devices()], 'origins': origins}))"""
    return json.loads(subprocess.check_output([*scientific_python,'-c',check],text=True,env=scientific_env))
runtime = runtime_info(); print(runtime)
assert runtime['jax_enable_x64'], 'JAX float64 must be enabled'
if ACCELERATOR == 'gpu': assert runtime['jax_backend'] == 'gpu', f"CUDA JAX device required, got {runtime['jax_backend']}"
subprocess.run([*scientific_python,'-m','pytest','-q'],cwd=repo,env=scientific_env,check=True)

In [ ]:
if RUN_SMOKE:
    subprocess.run([*scientific_python,'-m','energy_bo.experiments.run_task03a','--profile','smoke','--output-dir','artifacts/task03a/colab_smoke'],cwd=repo,env=scientific_env,check=True)
else: print('Smoke skipped; set RUN_SMOKE=True to run it.')

In [ ]:
if RUN_FULL:
    smoke_summary=repo/'artifacts/task03a/colab_smoke/SUMMARY.json'
    if not smoke_summary.exists(): raise RuntimeError('Run the smoke/preflight cell with RUN_SMOKE=True first.')
    command=[*scientific_python,'-m','energy_bo.experiments.run_task03a','--profile','full','--output-dir','artifacts/task03a/full']
    subprocess.run(command,cwd=repo,env=scientific_env,check=True)
    manifest={'git_sha':sha,'accelerator':ACCELERATOR,'runtime':runtime_info(),'command':command}
    out=repo/'artifacts/task03a/full'; (out/'colab_manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')
    archive=shutil.make_archive('/content/task03a_full_outputs','zip',root_dir=out)
    from google.colab import files; files.download(archive)
else: print('Full study did not run. Set RUN_FULL=True only after tests/preflight pass.')

## After download
Extract the ZIP locally into `artifacts/task03a/full/`. Do not upload raw output directly into Git. Review it first, then ask for an import audit; only compact reviewed evidence belongs under `results/task03a/full/`.